# Conversao ODBC para formato de fechamento

Converte qualquer export ODBC (CSV ou XLSX) para o layout padrao do fechamento geral.

## Como usar
1. Ajuste `BASE_PATH` para o arquivo ODBC de entrada (CSV ou XLSX).
2. Se for XLSX com varias abas, informe `BASE_SHEET` ou deixe `None` para auto-detectar a aba ODBC.
3. Ajuste `MESES_REFERENCIA` (`MM/AAAA`). Use `None` para converter **tudo** sem filtrar por mes.
4. Ajuste `CAMPO_DATA_FILTRO` se necessario (`lancamento` ou `ite_pagrec_vencimento`).
5. Execute todas as celulas.

## Mapeamento de valores
- `valor_nf` <- `valor_bruto` (nota fiscal valor cheio)
- `valor_pago` <- `iterea_valpago` (valor efetivamente pago)
- `valor_conta` <- `valor_centro` (rateio por centro de custo; sinal preservado)
- `Valor Oficial` <- `valor_centro` (nao usar `valor_plano`, que e rateio por plano de contas)

## Saida
- Por mes: `02-Referencias/Fechamento/FECHAMENTO_ODBC_{ano}_{mes}.xlsx`
- Consolidado (quando ha mais de um mes): `FECHAMENTO_ODBC_COMPLETO_{ano}.xlsx`
- Sem filtro de mes: `{stem_do_arquivo}_fechamento.xlsx`


In [ ]:
NOTEBOOK_VERSAO = '2.4'  # valor_nf<-valor_bruto; valor_pago<-iterea_valpago; valor_conta/Valor Oficial<-valor_centro

from pathlib import Path
import re
import pandas as pd
import numpy as np


def resolver_raiz_repo() -> Path:
    cwd = Path.cwd()
    for base in (cwd, cwd.parent, cwd.parent.parent):
        if (base / '02-Referencias').is_dir():
            return base
    raise FileNotFoundError(f'Nao foi possivel localizar 02-Referencias a partir de {cwd.resolve()}')


REPO_ROOT = resolver_raiz_repo()

# --- Parametros principais (ajuste aqui) ---
BASE_PATH = REPO_ROOT / '02-Referencias' / 'base_maio.xlsx'
BASE_SHEET = None          # None = auto-detecta aba ODBC no XLSX
MESES_REFERENCIA = ['05/2026']  # None = sem filtro por mes
CAMPO_DATA_FILTRO = 'lancamento'       # 'lancamento' ou 'ite_pagrec_vencimento'
OUT_DIR = REPO_ROOT / '02-Referencias' / 'Fechamento'
PREFIXO_SAIDA = 'FECHAMENTO_ODBC'      # prefixo dos arquivos gerados por mes

# Layout padrao do fechamento (nao depende de arquivo modelo externo)
COLUNAS_FECHAMENTO = [
    'id', 'Segmento',
    'n1_cod_centro_custo', 'n1_centro_custo', 'n1_CC',
    'n2_cod_centro_custo', 'n2_centro_custo', 'n2_CC',
    'n3_cod_centro_custo', 'n3_centro_custo', 'n3_CC',
    'n4_cod_centro_custo', 'n4_centro_custo', 'n4_CC',
    'cod_conta', 'conta', 'cod_conta-descr',
    'filial', 'titulo', 'valor_nf', 'valor_pago', 'valor_conta',
    'observacao', 'data_nf', 'data_pagamento',
    'cod_credor_forn_cli_func', 'credor_forn_cli_func',
    'Origem', 'Sistema', 'Dados auxiliares', 'Valor Oficial',
    'DE-PARA1', 'DE-PARA2', 'CUSTEIO VARIÁVEL',
]

COLUNAS_ODBC_OBRIGATORIAS = [
    'codcen', 'descen', 'codcdc', 'descdc', 'filial', 'documento',
    'valor_bruto', 'valor_plano', 'valor_centro', 'iterea_valpago',
    'observacao', 'lancamento', 'iterea_pagamento', 'codigo_pessoa', 'nome', 'nota',
]

# Colunas do Excel de saida com tipo/formato fixo (letras = layout padrao do fechamento)
# T/U/V = valor_nf, valor_pago, valor_conta | X/Y = data_nf, data_pagamento | AE = Valor Oficial
COLUNAS_NUMERICAS_SAIDA = ('valor_nf', 'valor_pago', 'valor_conta', 'Valor Oficial')
COLUNAS_DATA_SAIDA = ('data_nf', 'data_pagamento')
FMT_NUMERICO_EXCEL = '#.##0,00'
FMT_DATA_BR_EXCEL = 'DD/MM/YYYY'

BASE_PATH = Path(BASE_PATH)
OUT_DIR = Path(OUT_DIR)

if not BASE_PATH.exists():
    raise FileNotFoundError(f'Arquivo nao encontrado: {BASE_PATH.resolve()}')

print(f'Notebook v{NOTEBOOK_VERSAO} — sem arquivo MODELO externo')
print('Parametros carregados com sucesso.')
print(f'BASE={BASE_PATH.name} | MESES={MESES_REFERENCIA or "todos"} | CAMPO_DATA_FILTRO={CAMPO_DATA_FILTRO}')
print(f'Layout de saida: {len(COLUNAS_FECHAMENTO)} colunas (fixo no notebook)')


In [ ]:
def to_float_br(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v)
    s = str(v).strip().replace('R$', '').replace(' ', '')
    if s == '':
        return np.nan
    # BR com milhar: 1.234,56 | Excel/CSV US: 174.08 | BR simples: 174,08
    if ',' in s and '.' in s:
        s = s.replace('.', '').replace(',', '.')
    elif ',' in s:
        s = s.replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return np.nan

def to_br_number(v):
    if pd.isna(v):
        return ''
    s = f'{float(v):.2f}'
    return s.replace('.', ',')

def to_br_currency(v):
    if pd.isna(v):
        return ''
    txt = f'{float(v):,.2f}'
    txt = txt.replace(',', 'X').replace('.', ',').replace('X', '.')
    return f'R$ {txt}'

def split_descen(descen):
    partes = [p.strip() for p in str(descen).split('/') if p.strip()]
    natureza = partes[0] if len(partes) > 0 else ''
    divisao = partes[1] if len(partes) > 1 else ''
    filial_cc = partes[2] if len(partes) > 2 else ''
    resto = partes[3:] if len(partes) > 3 else []
    n3 = resto[0] if len(resto) > 0 else filial_cc
    n4 = ' / '.join(resto) if len(resto) > 0 else n3
    # ODBC as vezes repete o nome do nivel 3 no inicio de n4 (ex.: "PRENSA MOVEL / QXH2G14 (PHH0061)")
    if n3 and isinstance(n4, str) and n4.startswith(n3 + ' / '):
        n4 = n4[len(n3) + 3:].strip()
    # Quando n3 perde sufixo (ex.: JOINVILLE/SC), n4 pode ficar "SC / EHH0044"
    if isinstance(n4, str) and ' / ' in n4:
        tail = n4.split(' / ')[-1].strip()
        if re.match(
            r'^([A-Z]{3}\d{4}|[A-Z]{3}\d[A-Z0-9]{3}|[A-Z0-9]{5,8}\s*\([^)]+\))$',
            tail,
            re.IGNORECASE,
        ):
            n4 = tail
    if isinstance(n4, str) and ' / ' in n4 and 'operadores' in n4.lower() and 'arcelor' in n4.lower():
        n4 = 'ARCELOR RESENDE - OPERADORES'
    return natureza, divisao, filial_cc, n3, n4

def levels_from_codcen(codcen):
    tokens = [t for t in str(codcen).strip().split('.') if t]
    if len(tokens) < 2:
        n1 = str(codcen).strip()
    else:
        n1 = '.'.join(tokens[:2])
    n2 = '.'.join(tokens[:3]) if len(tokens) >= 3 else n1
    n3 = '.'.join(tokens[:4]) if len(tokens) >= 4 else n2
    n4 = '.'.join(tokens) if len(tokens) >= 1 else ''
    return n1, n2, n3, n4

def parse_data_nf(v):
    s = str(v).strip()
    if not s or set(s) == {'#'}:
        return pd.NaT
    dt = pd.to_datetime(s, format='%d/%m/%Y', errors='coerce')
    if pd.isna(dt):
        dt = pd.to_datetime(s, errors='coerce')
    return dt


def parse_data_pagamento(v):
    s = str(v).strip()
    if not s or s.upper() == 'NAO PAGO' or s in {'1800-01-01', '1900-01-01'}:
        return pd.NaT
    return pd.to_datetime(s, errors='coerce')


def fmt_data_nf(v):
    dt = parse_data_nf(v)
    if pd.isna(dt):
        return ''
    return dt.strftime('%d/%m/%Y')


def fmt_data_pagamento(v):
    dt = parse_data_pagamento(v)
    if pd.isna(dt):
        return ''
    return dt.strftime('%d/%m/%Y')


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str((Path.cwd().parent / "Utitlities").resolve()))
from fechamento_excel import gravar_fechamento_excel

def read_csv_with_fallback(path, sep=';', dtype=str, nrows=None):
    encodings = ['utf-8', 'utf-8-sig', 'cp1252', 'latin1']
    ultimo_erro = None
    for enc in encodings:
        try:
            df = pd.read_csv(path, sep=sep, dtype=dtype, encoding=enc, nrows=nrows, low_memory=False)
            return df, enc
        except UnicodeDecodeError as e:
            ultimo_erro = e
    raise UnicodeDecodeError(
        getattr(ultimo_erro, 'encoding', 'unknown'),
        getattr(ultimo_erro, 'object', b''),
        getattr(ultimo_erro, 'start', 0),
        getattr(ultimo_erro, 'end', 1),
        f'Nao foi possivel ler com encodings {encodings}: {ultimo_erro}'
    )


def normalizar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def validar_colunas_odbc(df: pd.DataFrame) -> None:
    faltantes = [c for c in COLUNAS_ODBC_OBRIGATORIAS if c not in df.columns]
    if faltantes:
        raise ValueError(
            'Colunas ODBC obrigatorias ausentes: '
            + ', '.join(faltantes)
            + f'\nColunas encontradas: {", ".join(df.columns)}'
        )


def detectar_aba_odbc(path: Path) -> str | int:
    xl = pd.ExcelFile(path)
    chave = {'codcen', 'valor_centro', 'valor_bruto'}
    for name in xl.sheet_names:
        cols = {str(c).strip() for c in pd.read_excel(path, sheet_name=name, nrows=0).columns}
        if chave.issubset(cols):
            return name
    return xl.sheet_names[0]


def read_odbc_base(path: Path, sheet: str | None = None):
    path = Path(path)
    if path.suffix.lower() in {'.xlsx', '.xls'}:
        sheet_name = sheet if sheet is not None else detectar_aba_odbc(path)
        if sheet is None:
            print(f'Aba ODBC auto-detectada: {sheet_name}')
        df = pd.read_excel(path, sheet_name=sheet_name, dtype=str)
        return normalizar_colunas(df), 'xlsx'
    df, enc = read_csv_with_fallback(path)
    return normalizar_colunas(df), enc


base, enc_base = read_odbc_base(BASE_PATH, BASE_SHEET)
validar_colunas_odbc(base)

print(f'Fonte ODBC: {BASE_PATH.name} ({enc_base}) | linhas: {len(base):,}'.replace(',', '.'))
print(f'Colunas ODBC: {len(base.columns)} | layout fechamento: {len(COLUNAS_FECHAMENTO)} colunas')


def filtrar_mes(base_df: pd.DataFrame, mes_referencia: str, campo_data: str) -> pd.DataFrame:
    mes, ano = mes_referencia.split('/')
    mes_int = int(mes)
    ano_int = int(ano)

    if campo_data == 'lancamento':
        serie_data = base_df[campo_data].fillna('').astype(str).str.strip()
        # Excel ODBC costuma vir em ISO (aaaa-mm-dd); CSV em dd/mm/aaaa
        iso_like = serie_data.str.match(r'^\d{4}-\d{2}-\d{2}', na=False).mean() > 0.5
        dt = pd.to_datetime(serie_data, errors='coerce') if iso_like else pd.to_datetime(serie_data, dayfirst=True, errors='coerce')
        filtro = (dt.dt.month == mes_int) & (dt.dt.year == ano_int)
        if not filtro.any():
            m2 = str(mes_int).zfill(2)
            padrao1 = rf'(^|\D){m2}/{ano_int}(\D|$)'
            padrao2 = rf'(^|\D){mes_int}/{ano_int}(\D|$)'
            filtro = serie_data.str.contains(padrao1, regex=True, na=False) | serie_data.str.contains(padrao2, regex=True, na=False)
    elif campo_data == 'ite_pagrec_vencimento':
        serie_data = base_df[campo_data].fillna('').astype(str).str.strip()
        dt = pd.to_datetime(serie_data, errors='coerce')
        filtro = (dt.dt.month == mes_int) & (dt.dt.year == ano_int)
        if not filtro.any():
            m2 = str(mes_int).zfill(2)
            filtro = serie_data.str.startswith(f'{ano_int}-{m2}')
    else:
        raise ValueError("campo_data deve ser 'lancamento' ou 'ite_pagrec_vencimento'.")

    resultado = base_df.loc[filtro].copy().reset_index(drop=True)
    if resultado.empty:
        raise ValueError(f"Nenhum registro encontrado para {mes_referencia} usando {campo_data}.")
    return resultado


def converter_para_fechamento(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work['valor_bruto_num'] = work['valor_bruto'].apply(to_float_br)
    work['valor_plano_num'] = work['valor_plano'].apply(to_float_br)
    work['valor_centro_num'] = work['valor_centro'].apply(to_float_br)
    work['iterea_valpago_num'] = work['iterea_valpago'].apply(to_float_br)

    out = pd.DataFrame(index=work.index)
    out['id'] = (work.index + 1).astype(str).str.zfill(6)

    lvl = work['codcen'].apply(levels_from_codcen)
    out['n1_cod_centro_custo'] = lvl.apply(lambda x: x[0])
    out['n2_cod_centro_custo'] = lvl.apply(lambda x: x[1])
    out['n3_cod_centro_custo'] = lvl.apply(lambda x: x[2])
    out['n4_cod_centro_custo'] = lvl.apply(lambda x: x[3])

    desc_split = work['descen'].apply(split_descen)
    out['n1_centro_custo'] = desc_split.apply(lambda x: x[1])
    out['n2_centro_custo'] = desc_split.apply(lambda x: x[2])
    out['n3_centro_custo'] = desc_split.apply(lambda x: x[3])
    out['n4_centro_custo'] = desc_split.apply(lambda x: x[4])

    out['Segmento'] = out['n1_centro_custo']
    out['n1_CC'] = (out['n1_cod_centro_custo'].fillna('') + ' ' + out['n1_centro_custo'].fillna('')).str.strip()
    out['n2_CC'] = (out['n2_cod_centro_custo'].fillna('') + ' ' + out['n2_centro_custo'].fillna('')).str.strip()
    out['n3_CC'] = (out['n3_cod_centro_custo'].fillna('') + ' ' + out['n3_centro_custo'].fillna('')).str.strip()
    out['n4_CC'] = (out['n4_cod_centro_custo'].fillna('') + ' ' + out['n4_centro_custo'].fillna('')).str.strip()

    out['cod_conta'] = work['codcdc'].fillna('')
    out['conta'] = work['descdc'].fillna('')
    out['cod_conta-descr'] = (out['cod_conta'] + ' ' + out['conta']).str.strip()
    out['filial'] = work['filial'].fillna('')
    out['titulo'] = work['documento'].fillna('')
    out['valor_nf'] = work['valor_bruto_num']
    out['valor_pago'] = work['iterea_valpago_num']
    out['valor_conta'] = work['valor_centro_num']
    out['observacao'] = work['observacao'].fillna('')
    out['data_nf'] = work['lancamento'].apply(parse_data_nf)
    out['data_pagamento'] = work['iterea_pagamento'].apply(parse_data_pagamento)
    out['cod_credor_forn_cli_func'] = work['codigo_pessoa'].fillna('')
    out['credor_forn_cli_func'] = work['nome'].fillna('')

    out['Origem'] = np.where(
        out['cod_conta'].str.startswith(('4.', '5.')),
        'Entrada (Origem)',
        'Saida (Aplicacoes)'
    )
    out['Sistema'] = 'SAGI'
    out['Dados auxiliares'] = work['nota'].fillna('')
    out['Valor Oficial'] = work['valor_centro_num']
    out['DE-PARA1'] = ''
    out['DE-PARA2'] = ''
    out['CUSTEIO VARIÁVEL'] = ''

    for c in COLUNAS_FECHAMENTO:
        if c not in out.columns:
            out[c] = ''

    result = out[COLUNAS_FECHAMENTO].copy()
    for c in COLUNAS_NUMERICAS_SAIDA:
        result[c] = pd.to_numeric(result[c], errors='coerce')
    for c in COLUNAS_DATA_SAIDA:
        result[c] = pd.to_datetime(result[c], errors='coerce')

    text_cols = [
        c for c in COLUNAS_FECHAMENTO
        if c not in COLUNAS_NUMERICAS_SAIDA and c not in COLUNAS_DATA_SAIDA
    ]
    result[text_cols] = result[text_cols].fillna('')
    return result


def validar_totais(df_origem: pd.DataFrame, df_saida: pd.DataFrame, rotulo: str = '') -> None:
    bruto = df_origem['valor_bruto'].apply(to_float_br).sum()
    valpago = df_origem['iterea_valpago'].apply(to_float_br).sum()
    centro = df_origem['valor_centro'].apply(to_float_br).sum()
    nf = pd.to_numeric(df_saida['valor_nf'], errors='coerce').sum()
    pago = pd.to_numeric(df_saida['valor_pago'], errors='coerce').sum()
    conta = pd.to_numeric(df_saida['valor_conta'], errors='coerce').sum()
    oficial = pd.to_numeric(df_saida['Valor Oficial'], errors='coerce').sum()
    prefixo = f'{rotulo}: ' if rotulo else ''
    print(f"{prefixo}ODBC valor_bruto={bruto:,.2f} | saida valor_nf={nf:,.2f}")
    print(f"{prefixo}ODBC iterea_valpago={valpago:,.2f} | saida valor_pago={pago:,.2f}")
    print(f"{prefixo}ODBC valor_centro={centro:,.2f} | saida valor_conta={conta:,.2f}")
    print(f"{prefixo}ODBC valor_centro={centro:,.2f} | saida Valor Oficial={oficial:,.2f}")


def salvar_fechamento(df_out: pd.DataFrame, caminho: Path) -> None:
    gravar_fechamento_excel(
        df_out,
        caminho,
        sheet_name='Fechamento',
        colunas_preservar_sinal=('valor_conta', 'Valor Oficial'),
    )

saidas_por_mes = {}
bases_por_mes = {}
arquivos_gerados: list[Path] = []

if MESES_REFERENCIA:
    lotes = [(mes_ref, filtrar_mes(base, mes_ref, CAMPO_DATA_FILTRO)) for mes_ref in MESES_REFERENCIA]
else:
    lotes = [(None, base.copy().reset_index(drop=True))]

for mes_ref, df_lote in lotes:
    out = converter_para_fechamento(df_lote)

    if mes_ref:
        mes, ano = mes_ref.split('/')
        chave = mes_ref
        caminho = OUT_DIR / f'{PREFIXO_SAIDA}_{ano}_{mes}.xlsx'
    else:
        chave = '__todos__'
        caminho = OUT_DIR / f'{BASE_PATH.stem}_fechamento.xlsx'

    saidas_por_mes[chave] = out
    bases_por_mes[chave] = df_lote
    salvar_fechamento(out, caminho)
    arquivos_gerados.append(caminho)
    validar_totais(df_lote, out, mes_ref or 'Todos')
    print(f'{(mes_ref or "Todos")}: {len(out):,} registros -> {caminho.name}'.replace(',', '.'))

if MESES_REFERENCIA and len(MESES_REFERENCIA) > 1:
    consolidado = pd.concat([saidas_por_mes[m] for m in MESES_REFERENCIA], ignore_index=True)
    base_consolidada = pd.concat([bases_por_mes[m] for m in MESES_REFERENCIA], ignore_index=True)
    ano_cons = MESES_REFERENCIA[0].split('/')[1]
    caminho_cons = OUT_DIR / f'{PREFIXO_SAIDA}_COMPLETO_{ano_cons}.xlsx'
    salvar_fechamento(consolidado, caminho_cons)
    arquivos_gerados.append(caminho_cons)
    validar_totais(base_consolidada, consolidado, 'Consolidado')
    print(f'Consolidado: {len(consolidado):,} registros -> {caminho_cons.name}'.replace(',', '.'))
    out = consolidado
else:
    out = next(iter(saidas_por_mes.values()))

display(out.head(5))

# --- Checagem de layout ---
faltantes = [c for c in COLUNAS_FECHAMENTO if c not in out.columns]
extras = [c for c in out.columns if c not in COLUNAS_FECHAMENTO]
print('Colunas faltantes:', faltantes)
print('Colunas extras:', extras)
print('Quantidade de colunas esperadas:', len(COLUNAS_FECHAMENTO))
print('Quantidade de colunas na saida:', len(out.columns))
print()
if arquivos_gerados:
    print('Arquivo(s) gerado(s):')
    for caminho in arquivos_gerados:
        print(f'  - {caminho.resolve()}')
else:
    print('Nenhum arquivo gerado.')


In [ ]:
# v2.4: valor_nf<-valor_bruto; valor_pago<-iterea_valpago; valor_conta/Valor Oficial<-valor_centro.
